# 8장. Omniverse에서 후크의 두께/하중에 따른 응력/변위 확인하기

## 개요

**목표:** 7장 서버와 Omniverse 연결 → 두께/하중 슬라이더로 형상/예측 갱신 → 응력/변위 크기 비교/저장

| 순서 | 실습에서 하는 일 |
|---|---|
| 1. 연결 준비 | 7장의 추론 서버가 준비됐는지 확인 |
| 2. 실행과 조작 | Omniverse를 열고 슬라이더/표시 모드/상태 익히기 |
| 3. 결과 해석 | 기준 조건의 응력/변위 크기와 최대 위치 읽기 |
| 4. 조건 비교 | 같은 하중에서 두께 비교 → 같은 두께에서 하중 비교 |
| 5. 저장과 정리 | 저장한 수치를 비교표로 읽고 실습 종료 |


## 1. 실습 준비와 추론 서버 연결 확인


### 1.2 서버 연결 확인하기


### 08장 / 셀 03 / Notebook


In [ ]:
from pathlib import Path
from datetime import datetime
import json, sys
working_dir = Path.cwd().resolve()
education_dir = next((p for base in [working_dir, *working_dir.parents]
                      for p in [base, base/'education']
                      if (p/'omniverse/kit_transolver.py').is_file()), None)
if education_dir is None:
    raise FileNotFoundError('교육 폴더 또는 그 하위 폴더에서 실행하세요.')
for path in [education_dir, education_dir/'omniverse']:
    if str(path) not in sys.path:
        sys.path.insert(0,str(path))
from course_utils.server_handoff import resolve_server_handoff
from course_utils.visualization_lesson import read_saved_results, pick_result, prepare_view_directory
from client import HookClient
connection_file, connection = resolve_server_handoff(education_dir)
api_url = connection['api_url']
client = HookClient(api_url)
health = client.request('/health')
assert health['ready'] and health['api_kind'] == 'live_transolver'
assert health['model_id'] == connection['model_sha256'] and health['device'] == 'cuda'
catalog = client.request('/geometries')
# 처음 실습할 때는 빈 문자열. 기존 기록을 읽을 때만 당시 실습 폴더 이름을 입력합니다.
RESUME_VIEW_FOLDER = ''
view_dir = prepare_view_directory(education_dir, health['model_id'], RESUME_VIEW_FOLDER)
kit_bat = view_dir/'start_omniverse.bat'
reference_thickness = catalog['reference_thickness_mm']
reference_force = catalog['nominal_force_N']
print('서버 준비:',health['ready'],'| 7장과 같은 모델:',health['model_id']==connection['model_sha256'])
print('API 주소:',api_url)
print('두께 범위(mm):',catalog['thickness_range_mm'],'| 하중 범위(N):',catalog['force_range_N'])
print('첫 요청 조건:',reference_thickness,'mm /',reference_force,'N')
print('이번 화면 실습 폴더:',view_dir)


## 2. Omniverse 실행과 화면 조작


### 2.1 실행용 BAT 만들고 실행하기


### 08장 / 셀 06 / Notebook


In [ ]:
from preparation.launch_kit import configured_kit, command
from course_utils.server_launcher import create_kit_launcher
kit_exe = configured_kit(education_dir)
command(kit_exe,transolver=True,api_url=api_url,expected_model_id=connection['model_sha256'])
kit_command = [sys.executable,str(education_dir/'preparation/launch_kit.py'),
               '--transolver','--api-url',api_url,'--expected-model-id',connection['model_sha256'],
               '--output-dir',str(view_dir/'kit')]
kit_bat = create_kit_launcher(view_dir/'start_omniverse.bat',kit_command,education_dir.parent)
print('Kit 실행 파일:',kit_exe)
print('Omniverse 실행 BAT:',kit_bat)
print('파일이 있는 폴더:',view_dir)
print('서버 Console을 유지하고, 파일 탐색기에서 BAT를 더블클릭하세요.')


## 3. 기준 조건의 응력과 변위 읽기


### 3.3 저장한 기준 결과 확인하기


### 08장 / 셀 09 / Notebook


In [ ]:
# Omniverse에서 Reference → Apply / AI stress [MPa] 선택 → LIVE 확인 → Save result 후 실행합니다.
records = read_saved_results(view_dir/'kit',connection['model_sha256'])
first_result = pick_result(records,reference_thickness,reference_force)
p = first_result['prediction']
print('저장 조건:',p['thickness_mm'],'mm /',p['force_N'],'N')
print('전체 최대응력(MPa):',p['full_max_stress_mpa'])
print('최대응력 위치(mm):',p['full_peak_position_mm'])
print('전체 최대 변위 크기(mm):',p['full_max_displacement_mm'])
print('최대 변위 위치(mm):',p['full_displacement_peak_position_mm'])
print('화면 기록:',first_result['record_path'])


## 5. 결과 저장과 실습 정리


### 5.1 비교에 필요한 기록 확인하기


### 08장 / 셀 13 / Notebook


In [ ]:
# 강의 화면의 네 비교 조건을 각각 AI stress [MPa] 모드로 저장한 뒤 실행합니다.
# Thin 6 mm / Thick 12 mm / Reference 10.25 mm: 같은 기준 하중 343.561 N. Higher force: 10.25 mm / 400 N.
records = read_saved_results(view_dir/'kit',connection['model_sha256'])
comparison_results = [pick_result(records,t,f) for t,f in
                      [(6.,reference_force),(12.,reference_force),
                       (reference_thickness,reference_force),(reference_thickness,400.)]]
print('두께(mm) | 하중(N) | 전체 최대응력(MPa) | 전체 최대 변위 크기(mm) | 학습 영역 구분')
for row in comparison_results:
    p = row['prediction']
    scope = '학습 영역 안' if p['inside_training_hull'] else '외삽(학습 영역 밖)'
    print(f"{p['thickness_mm']:g} | {p['force_N']:.3f} | {p['full_max_stress_mpa']:.3f} | "
          f"{p['full_max_displacement_mm']:.6f} | {scope}")
thin,thick = comparison_results[:2]
assert thin['display']['color_min']==thick['display']['color_min']
assert thin['display']['color_max']==thick['display']['color_max']
print('두께 비교의 공통 응력 색 범위:',thin['display']['color_min'],thick['display']['color_max'],'MPa')


### 5.2 이번 실습의 기록 모으기


### 08장 / 셀 16 / Notebook


In [ ]:
records = read_saved_results(view_dir/'kit',connection['model_sha256'])
lesson_record = dict(model_sha256=connection['model_sha256'],api_url=api_url,
                    connection_file=str(connection_file),kit_launcher=str(kit_bat),
                    saved_results=[dict(record_path=r['record_path'],condition=r['input'],
                                        display_mode=r['display']['mode'],prediction=r['prediction']) for r in records],
                    purpose='Omniverse visualization of Chapter 7 responses; not a new ANSYS accuracy evaluation')
lesson_record_file = view_dir/'visualization_lesson_record.json'
lesson_record_file.write_text(json.dumps(lesson_record,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print('저장한 화면 결과 수:',len(records))
print('이번 실습 기록:',lesson_record_file)
print('화면/USD/조건 기록:',view_dir/'kit')
